In [6]:
import pandas as pd
movies=pd.read_csv("movies.csv", encoding='latin1')#converting utf to latini encoding
ratings=pd.read_csv("ratings.csv", encoding='latin1')

In [7]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [8]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [12]:
#converting the ids to categorical data for the Neural Network to understand
ratings['userId']=ratings['userId'].astype('category').cat.codes
ratings['movieId']=ratings['movieId'].astype('category').cat.codes

In [11]:
from sklearn.model_selection import train_test_split
X=ratings[['userId','movieId']]
Y=ratings['rating']
X_train, X_test, y_train, y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [13]:
print("Users:",ratings['userId'].nunique())
print("Movies:",ratings['movieId'].nunique())
print(ratings.head())

Users: 610
Movies: 9724
   userId  movieId  rating  timestamp
0       0        0     4.0  964982703
1       0        2     4.0  964981247
2       0        5     4.0  964982224
3       0       43     5.0  964983815
4       0       46     5.0  964982931


In [21]:
import tensorflow as tf
from tensorflow.keras import layers,models

num_users=ratings['userId'].nunique()
num_movies=ratings['movieId'].nunique()

user_input=layers.Input(shape=(1,))
movie_input=layers.Input(shape=(1,))

user_embedding=layers.Embedding(num_users,32)(user_input)
movie_embedding=layers.Embedding(num_movies,32)(movie_input)

user_vec=layers.Flatten()(user_embedding)
movie_vec=layers.Flatten()(movie_embedding)

x=layers.Concatenate()([user_vec,movie_vec])
x=layers.Dense(64,activation='relu')(x)
x=layers.Dense(32,activation='relu')(x)
output=layers.Dense(1)(x)

model=models.Model([user_input,movie_input],output)

model.compile(optimizer='adam',loss='mean_squared_error',metrics=['mae'])

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 32)     │     19,520 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 32)     │    311,168 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 32)        │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 32)        │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 64)        │          0 │ flatten_4[0][0],  │
│ (Concatenate)       │                   │            │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      4,160 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      2,080 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         33 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 336,961 (1.29 MB)

 Trainable params: 336,961 (1.29 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history=model.fit([X_train['userId'],X_train['movieId']],y_train,epochs=10,batch_size=64,validation_split=0.1)


Epoch 1/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 2.8906 - mae: 1.2222 - val_loss: 0.8093 - val_mae: 0.6898
Epoch 2/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.7037 - mae: 0.6417 - val_loss: 0.7732 - val_mae: 0.6755
Epoch 3/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.6512 - mae: 0.6153 - val_loss: 0.7579 - val_mae: 0.6659
Epoch 4/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.6074 - mae: 0.5926 - val_loss: 0.7732 - val_mae: 0.6677
Epoch 5/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.5632 - mae: 0.5683 - val_loss: 0.7801 - val_mae: 0.6713
Epoch 6/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.5245 - mae: 0.5453 - val_loss: 0.7785 - val_mae: 0.6749
Epoch 7/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4774 - mae: 0.5178 - val_loss: 0.8122 - val_mae: 0.6897
Epoch 8/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.4295 - mae: 0.4884 - val_loss: 0.8432 - val_mae: 0.6993
Epoch 9/10
1135/1135 ━━━━━━━━━━━━━━━━━━━

In [23]:
model.evaluate([X_test['userId'],X_test['movieId']],y_test)

631/631 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.8875 - mae: 0.7139


[0.8839048743247986, 0.7127937078475952]

In [25]:
movie_id_map=dict(zip(movies['movieId'].astype('category').cat.codes,movies['title']))

In [27]:
import numpy as np

def recommend_movies(user_id,top_n=5):
  user_ids=np.array([user_id]*num_movies)
  movie_ids=np.array(range(num_movies))

  predictions=model.predict([user_ids,movie_ids],verbose=0)

  top_indices=predictions.flatten().argsort()[-top_n:][::-1]

  for idx in top_indices:
    print(movie_id_map[idx],"-> predicted rating :",round(predictions[idx][0],2))

In [29]:
recommend_movies(user_id=5,top_n=10)

Maz Jobrani: Immigrant (2017) -> predicted rating : 5.66
Thing: Terror Takes Shape, The (1998) -> predicted rating : 5.53
Jesus Christ Superstar (1973) -> predicted rating : 5.5
Letter to Three Wives, A (1949) -> predicted rating : 5.49
Jane Eyre (2011) -> predicted rating : 5.45
Sonatine (Sonachine) (1993) -> predicted rating : 5.44
Good, the Bad, the Weird, The (Joheunnom nabbeunnom isanghannom) (2008) -> predicted rating : 5.41
That Demon Within (2014) -> predicted rating : 5.37
Bart Got a Room (2008) -> predicted rating : 5.35
Get Him to the Greek (2010) -> predicted rating : 5.35
